# Loan Roll Rate Analysis

Analyze loan data to classify DPD (Days Past Due) into buckets and track month-over-month status changes.

In [3]:
import pandas as pd

## Read loan data

In [4]:
df = pd.read_csv("loan.csv", parse_dates=["SnapshotDate"])
df

,SnapshotDate,CustomerID,Age,Gender,CreditLimit,Outstanding,DayPastDue
0,2025-01-01,C001,35,M,50000,27788.54,0
1,2025-02-01,C001,35,M,50000,16750.22,5
2,2025-03-01,C001,35,M,50000,23000.59,10
3,2025-04-01,C001,35,M,50000,19464.21,0
4,2025-05-01,C001,35,M,50000,29729.42,0
...,...,...,...,...,...,...,...
295,2025-08-01,C025,53,M,110000,90273.94,100
296,2025-09-01,C025,53,M,110000,84818.22,75
297,2025-10-01,C025,53,M,110000,60925.47,50
298,2025-11-01,C025,53,M,110000,50060.73,25


## DPD Bucket mapping

In [5]:
def dpd_bucket(dpd):
    if dpd <= 30:
        return "Normal"
    elif dpd <= 60:
        return "Mild"
    elif dpd <= 90:
        return "Moderate"
    elif dpd <= 120:
        return "Severe"
    else:
        return "Red Zone"

df["DPD_Bucket"] = df["DayPastDue"].apply(dpd_bucket)
df

,SnapshotDate,CustomerID,Age,Gender,CreditLimit,Outstanding,DayPastDue,DPD_Bucket
0,2025-01-01,C001,35,M,50000,27788.54,0,Normal
1,2025-02-01,C001,35,M,50000,16750.22,5,Normal
2,2025-03-01,C001,35,M,50000,23000.59,10,Normal
3,2025-04-01,C001,35,M,50000,19464.21,0,Normal
4,2025-05-01,C001,35,M,50000,29729.42,0,Normal
...,...,...,...,...,...,...,...,...
295,2025-08-01,C025,53,M,110000,90273.94,100,Severe
296,2025-09-01,C025,53,M,110000,84818.22,75,Moderate
297,2025-10-01,C025,53,M,110000,60925.47,50,Mild
298,2025-11-01,C025,53,M,110000,50060.73,25,Normal


## Bucket order for comparison (higher = worse)

In [6]:
bucket_order = {
    "Normal": 0,
    "Mild": 1,
    "Moderate": 2,
    "Severe": 3,
    "Red Zone": 4,
}

df["_bucket_rank"] = df["DPD_Bucket"].map(bucket_order)

## Compute DPD Status (vs previous month)

In [9]:
df = df.sort_values(["CustomerID", "SnapshotDate"]).reset_index(drop=True)

df["_prev_rank"] = df.groupby("CustomerID")["_bucket_rank"].shift(1)

def dpd_status(row):
    if pd.isna(row["_prev_rank"]):
        return "New"          # first snapshot, no prior month
    elif row["_bucket_rank"] > row["_prev_rank"]:
        return "Worsen"       # bucket worsened
    elif row["_bucket_rank"] < row["_prev_rank"]:
        return "Improve"      # bucket improved
    else:
        return "NoChange"     # same bucket

df["DPD_Status"] = df.apply(dpd_status, axis=1)
df

,SnapshotDate,CustomerID,Age,Gender,CreditLimit,Outstanding,DayPastDue,DPD_Bucket,_bucket_rank,_prev_rank,DPD_Status
0,2025-01-01,C001,35,M,50000,27788.54,0,Normal,0,NaN,New
1,2025-02-01,C001,35,M,50000,16750.22,5,Normal,0,0.0,NoChange
2,2025-03-01,C001,35,M,50000,23000.59,10,Normal,0,0.0,NoChange
3,2025-04-01,C001,35,M,50000,19464.21,0,Normal,0,0.0,NoChange
4,2025-05-01,C001,35,M,50000,29729.42,0,Normal,0,0.0,NoChange
...,...,...,...,...,...,...,...,...,...,...,...
295,2025-08-01,C025,53,M,110000,90273.94,100,Severe,3,4.0,Improve
296,2025-09-01,C025,53,M,110000,84818.22,75,Moderate,2,3.0,Improve
297,2025-10-01,C025,53,M,110000,60925.47,50,Mild,1,2.0,Improve
298,2025-11-01,C025,53,M,110000,50060.73,25,Normal,0,1.0,Improve
